In [34]:
%pip uninstall onnxruntime
%pip install --upgrade onnxruntime-gpu onnxscript
%pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 13.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 20.3 MB/s eta 0:00:00


In [16]:
%pip show transformers

Name: transformers
Version: 5.9.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: optimum, optimum-onnx, peft, sentence-transformers


In [57]:
%%capture

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "pameydorke/redred-gemma-4-E2B-it"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model.eval()

In [58]:
from transformers.cache_utils import DynamicCache

def tuple_to_dynamic_cache(past_key_values: tuple) -> DynamicCache:
    cache = DynamicCache()
    cache.key_cache   = [kv[0] for kv in past_key_values]
    cache.value_cache = [kv[1] for kv in past_key_values]
    return cache

def dynamic_cache_to_tuple(cache: DynamicCache) -> tuple:
    return tuple(zip(cache.key_cache, cache.value_cache))

In [59]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Optional, Tuple
from torch.export import Dim
from transformers.cache_utils import DynamicCache

config     = model.config
text_cfg   = config.text_config 

num_layers   = text_cfg.num_hidden_layers        # 35
num_kv_heads = text_cfg.num_key_value_heads      # 1
head_dim     = text_cfg.head_dim                 # 256
hidden_size  = text_cfg.hidden_size              # 1536
vocab_size   = text_cfg.vocab_size

class NativeGemma4Wrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model                     # Gemma4ForCausalLM

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: torch.LongTensor,
        position_ids: torch.LongTensor,
        past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
    ):
        cache = tuple_to_dynamic_cache(past_key_values)

        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=cache,
            use_cache=True,
        )

        presents = dynamic_cache_to_tuple(out.past_key_values)
        return out.logits, presents

In [60]:
B, S, P = 1, 1, 0

dummy_input_ids      = torch.randint(0, text_cfg.vocab_size, (B, S), dtype=torch.long)
dummy_attention_mask = torch.ones(B, P + S, dtype=torch.long)
dummy_position_ids   = torch.arange(P, P + S, dtype=torch.long).unsqueeze(0)

# Always pass tensors, never None, so Dynamo doesn’t hit control-flow branches.
# Shape per layer: [batch, num_kv_heads, past_len, head_dim]
dummy_past_kv = tuple(
    (
        torch.zeros(B, num_kv_heads, P, head_dim, dtype=torch.float32),
        torch.zeros(B, num_kv_heads, P, head_dim, dtype=torch.float32),
    )
    for _ in range(num_layers)
)

In [ ]:
model = model.cpu().eval()
wrapper = NativeGemma4Wrapper(model).cpu().eval()

In [62]:
dummy_input_ids      = dummy_input_ids.cpu()
dummy_attention_mask = dummy_attention_mask.cpu()
dummy_position_ids   = dummy_position_ids.cpu()
dummy_past_kv        = tuple(
    (k.cpu(), v.cpu())
    for k, v in dummy_past_kv
)

In [76]:
import torch.nn.functional as F

_orig_sdpa = F.scaled_dot_product_attention

def _sdpa_export(query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None):
    # SymBool / symbolic -> concrete bool
    if not isinstance(is_causal, bool):
        is_causal = False   # attention_mask already handles causality
    return _orig_sdpa(query, key, value, attn_mask=attn_mask, dropout_p=dropout_p, is_causal=is_causal, scale=scale)

F.scaled_dot_product_attention = _sdpa_export

In [77]:

batch_dim = Dim("batch")
seq_dim   = Dim("seq_len")
total_dim = Dim("total_len")
past_dim  = Dim("past_len")   # dynamic cache length

# Match the structure of (input_ids, attention_mask, position_ids, past_key_values)
dynamic_shapes = (
    {0: batch_dim, 1: seq_dim},           # input_ids
    {0: batch_dim, 1: total_dim},         # attention_mask
    {0: batch_dim, 1: seq_dim},           # position_ids
    tuple(                                # past_key_values: tuple of 35 layers
        (
            {0: batch_dim, 2: past_dim},  # key tensor
            {0: batch_dim, 2: past_dim},  # value tensor
        )
        for _ in range(num_layers)
    ),
)

In [82]:
import sys

old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(50000)

try:
    torch.onnx.export(
        wrapper,
        (dummy_input_ids, dummy_attention_mask, dummy_position_ids, dummy_past_kv),
        "decoder_model_merged.onnx",
        dynamo=True,
        opset_version=21,
        input_names=["input_ids", "attention_mask", "position_ids", "past_key_values"],
        output_names=["logits", "present_key_values"],
        dynamic_shapes=dynamic_shapes,
        external_data=True,
    )
finally:
    sys.setrecursionlimit(old_limit)   # always restore
    F.scaled_dot_product_attention = _orig_sdpa

[torch.onnx] Obtain model graph for `NativeGemma4Wrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NativeGemma4Wrapper([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `NativeGemma4Wrapper([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `NativeGemma4Wrapper([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'RecursionError'>: maximum recursion depth exceeded

(Refer to the full stack trace above for more information.)